In [92]:
import pandas as pd
import numpy as np

In [93]:
### DEFINE BETA VALUES FOR VDF

beta_vals = {} # TBD: compute values based on growth

beta_template_aux = {
    "Period": ["AM", "MD", "PM", "NT"],
    1: [2.542, 2.545, 2.663, 2.545],
    2: [0.715, 0.708, 0.592, 0.708],
    3: [3.130, 3.133, 3.283, 3.286]
}

beta_vals_aux = pd.DataFrame(beta_template_aux)

beta_vals_aux.set_index("Period", inplace=True)

beta_vals[2025] = beta_vals_aux

In [94]:
### HERE ARE THE MEASURED SPEEDS WE USE AS REFERENCE

measured_speeds_file = r"inputs/measured_speeds.csv"

measured_speeds = pd.read_csv(
    measured_speeds_file,
    sep=",",          # `delimiter` y `sep` son equivalentes; elige uno
    encoding="utf-8",
    decimal=".",      # parsea decimales con punto
    thousands=",",    # parsea separador de miles con coma
    quotechar='"',
    index_col=0
)

measured_speeds

,1NB,2NB,3NB,4NB,5NB,6NB,7NB,1SB,2SB,3SB,4SB,5SB,6SB,7SB
Capacity Factors,,,,,,,,,,,,,,
Night,67.990810,59.385031,62.305540,63.902308,64.165910,66.487727,66.982424,68.593430,65.916472,66.688172,64.697718,67.292700,66.050793,68.151660
AM-Early,70.319291,64.810802,70.239916,66.748852,65.394799,66.956909,68.172258,71.655422,66.809941,65.941932,66.154604,70.180779,68.845519,71.864047
AM-Peak,69.203112,60.145956,61.709257,62.158854,62.732156,62.513974,65.947224,64.334782,43.394233,37.685598,16.559854,36.154433,58.160010,66.969362
AM-Shoulder,69.203112,57.644223,56.397013,61.330126,61.165285,60.789826,66.204030,59.876449,31.814258,30.615897,14.838968,35.395523,51.963859,63.819564
MD,67.912208,52.868253,51.499039,61.067705,63.115607,60.210613,66.135436,67.474820,41.483567,46.069721,38.395470,59.477969,64.282166,68.490264
PM-Shoulder,48.986473,28.014015,40.880374,56.150217,62.725624,46.354783,66.030721,67.245858,47.532765,55.907290,58.262476,66.007867,63.974374,67.871600
PM-Peak,18.528380,18.265478,27.197959,43.973019,37.504650,24.812425,57.531655,67.606710,60.462434,51.310820,51.969905,62.551745,61.940654,67.338415
PM-Late,38.133675,27.162696,35.545576,53.462383,56.620362,50.559621,60.895541,68.669051,62.320094,57.552400,57.750710,65.911111,63.147594,67.529190


In [95]:
### DEFINE LOOKUP TABLE FOR BONUS PER PERIOD

lookup_period_file = r"inputs/LookUp_Period.csv"

lookup_period = pd.read_csv(
    lookup_period_file,
    sep=",",          # `delimiter` y `sep` son equivalentes; elige uno
    encoding="utf-8",
    decimal=".",      # parsea decimales con punto
    thousands=",",    # parsea separador de miles con coma
    quotechar='"',
    index_col=0
)

# Clip y reasignar
lookup_period = lookup_period*0

lookup_period

,Bonus/Mile,4 Periods
Period,,
Night,0.0,
AM-Early,0.0,
AM-Peak,0.0,
AM-Shoulder,0.0,
MD,0.0,
PM-Shoulder,0.0,
PM-Peak,0.0,
PM-Late,0.0,


In [ ]:
### DEFINE SEGMENT PARAMETERS
# Default configuration for time periods in traffic data

#TBD: Make this automatically
period_template = [                 # (Period, Hours/Day, Peak/OP, 4Periods tag)
    ("Night",        8, "OP",   "NT"),
    ("AM-Early",     1, "OP",   "AM"),
    ("AM-Peak",      2, "Peak", "AM"),
    ("AM-Shoulder",  1, "OP",   "AM"),
    ("MD",           5, "OP",   "MD"),
    ("PM-Shoulder",  1, "OP",   "PM"),
    ("PM-Peak",      3, "Peak", "PM"),
    ("PM-Late",      3, "OP",   "PM"),
]

rows = []
years = [2025]

# Default time periods list (for reference)
default_time_periods = [
    "Night",
    "AM-Early",
    "AM-Peak",
    "AM-Shoulder",
    "MD",
    "PM-Shoulder",
    "PM-Peak",
    "PM-Late"
]

# Create the base scenario: hour -> time period mapping
hour_to_period = {
    0: "Night",
    1: "Night",
    2: "Night",
    3: "Night",
    4: "Night",
    5: "Night",
    6: "AM-Early",
    7: "AM-Peak",
    8: "AM-Peak",
    9: "AM-Shoulder",
    10: "MD",
    11: "MD",
    12: "MD",
    13: "MD",
    14: "MD",
    15: "PM-Shoulder",
    16: "PM-Peak",
    17: "PM-Peak",
    18: "PM-Peak",
    19: "PM-Late",
    20: "PM-Late",
    21: "PM-Late",
    22: "Night",
    23: "Night"
}

period_to_period = {
    'Evening': 'Night',
    'Evening': 'PM-Late',
    'EarlyAM': 'AM-Early',
    'AM': 'AM-Peak',
    'AM': 'AM-Shoulder',
    'Midday': 'MD',
    'Midday': 'PM-Shoulder',
    'PM': 'PM-Peak'
}

# Define the segments and their parameters

awt_adt = 1.1 # Average weekday traffic (AWT) to average daily traffic (ADT) ratio
peak_factor = 1 # 1.05 # Peak factor for adjustment at peak hour traffic

hov_percentage = pd.DataFrame({
    'Year' : [2025],
    'HOV percentage' : [0]
})

hov_percentage.set_index('Year', inplace=True)

"""
S1: Chamblee - I 285
S2: I 285 - Jimmy
S3: Jimmy - Indian Trail
S4: Indian Trail -  GA-316
S5: GA-316 - Old Peachtree
S6: Old Peachtree - I-985
S7: I-985 - Hamilton Mill
"""

lengths = [3,3,2.2,2.2,3.4,3.4,4.4,4.4,3,3,4.4,4.4,5.6,5.6]
inscope = [0.95]*14

# Define segment parameters base
seg_params = pd.DataFrame({
    'SegDir':   ["1NB","1SB","2NB","2SB","3NB","3SB","4NB","4SB","5NB","5SB","6NB","6SB","7NB","7SB"],
    'Length':    lengths,
    'Inscope':   inscope,
    'Lanes_GP':  [8,8,8,8,8,8,8,8,6,6,7,7,5,5],# [6,6,6,6,6,6,6,6,4,4,5,5,3,3], # We may need to sum the toll lane
    'Lanes_ML':  [1]*14, # Lanes_ML': [2,2,2,2,2,2,2,2,3,3,2,2,2,2], # Do test changing segment 5
    'CapPerLane_GP': [2000]*14,
    'CapPerLane_ML': [1800]*14, #[1800]*26,
    'Speed_GP':  [65]*2 + [70]*12,
    'Speed_ML':  [70]*14,
    'Alpha_GP':  [1]*14,
    'Beta_GP':   [6]*14,
    'Alpha_ML':  [1]*14,
    'Beta_ML':   [6]*14,
    'Min_Toll_2016': [None]*14,
    'Max_Toll_2016': [None]*14,
    'LanesGP_AM_Peak': [5]*14,
    'LanesGP_PM_Peak': [5]*14,
})

seg_params.set_index('SegDir', inplace=True)

# Compute capacities as lanes * cap per lane
seg_params['Cap_GP'] = seg_params['Lanes_GP'] * seg_params['CapPerLane_GP']
seg_params['Cap_ML'] = seg_params['Lanes_ML'] * seg_params['CapPerLane_ML']

# Compute peak capacities as Alpha * base capacity
seg_params['CapGP_Peak'] = seg_params['Alpha_GP'] * seg_params['Cap_GP']
seg_params['CapML_Peak'] = seg_params['Alpha_ML'] * seg_params['Cap_ML']

# Optional: if you want integer capacities
seg_params[['Cap_GP','Cap_ML','CapGP_Peak','CapML_Peak']] = seg_params[
    ['Cap_GP','Cap_ML','CapGP_Peak','CapML_Peak']
].astype(int)

# Preview
seg_params

,Length,Inscope,Lanes_GP,Lanes_ML,CapPerLane_GP,CapPerLane_ML,Speed_GP,Speed_ML,Alpha_GP,Beta_GP,Alpha_ML,Beta_ML,Min_Toll_2016,Max_Toll_2016,LanesGP_AM_Peak,LanesGP_PM_Peak,Cap_GP,Cap_ML,CapGP_Peak,CapML_Peak
SegDir,,,,,,,,,,,,,,,,,,,,
1NB,3.0,0.95,8,2,2000,1800,65,70,1,6,1,6,None,None,5,5,16000,3600,16000,3600
1SB,3.0,0.95,8,2,2000,1800,65,70,1,6,1,6,None,None,5,5,16000,3600,16000,3600
2NB,2.2,0.95,8,2,2000,1800,70,70,1,6,1,6,None,None,5,5,16000,3600,16000,3600
2SB,2.2,0.95,8,2,2000,1800,70,70,1,6,1,6,None,None,5,5,16000,3600,16000,3600
3NB,3.4,0.95,8,2,2000,1800,70,70,1,6,1,6,None,None,5,5,16000,3600,16000,3600
3SB,3.4,0.95,8,2,2000,1800,70,70,1,6,1,6,None,None,5,5,16000,3600,16000,3600
4NB,4.4,0.95,8,2,2000,1800,70,70,1,6,1,6,None,None,5,5,16000,3600,16000,3600
4SB,4.4,0.95,8,2,2000,1800,70,70,1,6,1,6,None,None,5,5,16000,3600,16000,3600
5NB,3.0,0.95,6,2,2000,1800,70,70,1,6,1,6,None,None,5,5,12000,3600,12000,3600


In [97]:
import numpy as np

def adjusted_cumprod(row, target_year, multiplier):
    years = row.index
    print(row.values)
    factors = 1 + row.values
    
    # Find the index of the target year
    target_idx = list(years).index(target_year)
    
    # Apply multiplier to the target year's factor
    factors[target_idx] *= multiplier
    
    # Calculate cumulative product
    return pd.Series(np.cumprod(factors), index=years)

In [98]:
### IMPORT GROWTHS FOR EACH CLASS
file_path_growths = r"inputs/growths_per_segment.csv"
base_growth_df = pd.read_csv(
    file_path_growths,
    delimiter=',',
    encoding='utf-8',
    decimal='.',        # ← this tells pandas how to parse decimals
    thousands=',',       # ← this tells pandas how to parse thousands
    quotechar='"'
)

base_growth_df = base_growth_df.iloc[:, 1:]
project_years = base_growth_df.columns[1:].tolist()
base_growth_df.iloc[:, 1:] =  base_growth_df.iloc[:, 1:] + 1

base_growth_df.loc[:, '2032'] *= 1.12

base_growth_df

,SegmentMapped,2025,2026,2027,2028,2029,2030,2031,2032,2033,...,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054
0,S1,1,1.014281,1.014281,1.014281,1.014281,1.016553,1.016553,1.138540,1.016553,...,1.014706,1.014706,1.014706,1.014706,1.014706,1.013360,1.013360,1.013360,1.013360,1.013360
1,S2,1,1.013951,1.013951,1.013951,1.013951,1.016255,1.016255,1.138206,1.016255,...,1.014505,1.014505,1.014505,1.014505,1.014505,1.013242,1.013242,1.013242,1.013242,1.013242
2,S3,1,1.014263,1.014263,1.014263,1.014263,1.016485,1.016485,1.138464,1.016485,...,1.014751,1.014751,1.014751,1.014751,1.014751,1.013492,1.013492,1.013492,1.013492,1.013492
3,S4,1,1.014321,1.014321,1.014321,1.014321,1.016514,1.016514,1.138496,1.016514,...,1.014982,1.014982,1.014982,1.014982,1.014982,1.013747,1.013747,1.013747,1.013747,1.013747
4,S5,1,1.014252,1.014252,1.014252,1.014252,1.016410,1.016410,1.138380,1.016410,...,1.015084,1.015084,1.015084,1.015084,1.015084,1.013887,1.013887,1.013887,1.013887,1.013887
5,S6,1,1.014025,1.014025,1.014025,1.014025,1.016204,1.016204,1.138149,1.016204,...,1.015290,1.015290,1.015290,1.015290,1.015290,1.014162,1.014162,1.014162,1.014162,1.014162
6,S7,1,1.013925,1.013925,1.013925,1.013925,1.016081,1.016081,1.138011,1.016081,...,1.015273,1.015273,1.015273,1.015273,1.015273,1.014151,1.014151,1.014151,1.014151,1.014151
7,S8,1,1.013881,1.013881,1.013881,1.013881,1.016023,1.016023,1.137946,1.016023,...,1.015348,1.015348,1.015348,1.015348,1.015348,1.014225,1.014225,1.014225,1.014225,1.014225
8,S9,1,1.013836,1.013836,1.013836,1.013836,1.015960,1.015960,1.137875,1.015960,...,1.015339,1.015339,1.015339,1.015339,1.015339,1.014214,1.014214,1.014214,1.014214,1.014214
9,S10,1,1.016112,1.016112,1.016112,1.016112,1.016112,1.018178,1.140359,1.018178,...,1.015975,1.015975,1.015975,1.015975,1.015975,1.014326,1.014326,1.014326,1.014326,1.014326


In [99]:
### IMPORT COUNTS AND SEPARATE BY CLASS AND PERIODS

file_path_counts = r"inputs/counts_by_hour_grouped_sorted.csv"
base_counts_df = pd.read_csv(
    file_path_counts,
    delimiter=',',
    encoding='utf-8',
    decimal='.',        # ← this tells pandas how to parse decimals
    thousands=',',       # ← this tells pandas how to parse thousands
    quotechar='"'
)

# --- Ajustar direcciones ---
base_counts_df["Direction"] = base_counts_df["Direction"].replace({"EB": "EB", "WB": "WB"})

# --- Crear columna Seg/Dir ---
base_counts_df["Seg/Dir"] = base_counts_df["Segment"].astype(str) + base_counts_df["Direction"]

# --- Función para procesar cada clase ---
def process_class(df_class):
    # Convertir a formato largo
    df_long = df_class.melt(
        id_vars=["Seg/Dir", "Segment", "Direction", "Class"],
        value_vars=[str(h) for h in range(24)],
        var_name="Hour",
        value_name="Volume"
    )
    
    # Mapear hora a periodo
    df_long["Hour"] = df_long["Hour"].astype(int)
    df_long["Period"] = df_long["Hour"].map(hour_to_period)
    
    # Agregar por Segment/Direction/Class/Period
    df_period = df_long.groupby(
        ["Seg/Dir", "Segment", "Direction", "Class", "Period"], as_index=False, sort=False
    ).agg({"Volume": "mean"}).round(0)
    
    # Pivot a formato ancho (periodos como columnas)
    period_order = df_period['Period'].unique()
    df_wide = df_period.pivot(
        index=["Seg/Dir", "Segment", "Direction", "Class"],
        columns="Period",
        values="Volume"
    )[period_order].reset_index()
    
    # Mantener solo Seg/Dir como índice
    df_proc = df_wide.drop(columns=["Class", "Direction", "Segment"]).set_index("Seg/Dir")
    
    return df_proc

# --- Separar por clases y procesar ---
dfs_by_class = {}
for cls in base_counts_df["Class"].unique():
    df_cls = base_counts_df[base_counts_df["Class"] == cls].copy()
    dfs_by_class[cls] = process_class(df_cls)


'''
Vehicle Classifications follow FHWA standards:
Lights: FHWA Classes 1-3 [Light Duty Vehicles]
Medium A: Classes 4-5 [Buses and Single Unit 2 axles trucks] 
Medium B: Class 6-7 [Single Unit 3 or 4 axles Trucks]
Heavy A: Classes 8-10 [Single Trailer 3 or more axles trucks]
Heavy B: Classes 11-13 [Combination Trucks Multitrailer Trucks]
'''

# --- Ejemplo de uso ---
df_lights = dfs_by_class["Lights"]
df_mediumA = dfs_by_class["Medium A"]
df_mediumB = dfs_by_class["Medium B"]
df_heavyA = dfs_by_class["Heavy A"]
df_heavyB = dfs_by_class["Heavy B"]

df_lights

Period,Night,AM-Early,AM-Peak,AM-Shoulder,MD,PM-Shoulder,PM-Peak,PM-Late
Seg/Dir,,,,,,,,
S1NB,1830.0,3467.0,4855.0,4132.0,5636.0,7121.0,6508.0,5882.0
S1SB,1785.0,7354.0,7993.0,6891.0,5561.0,6430.0,6004.0,3804.0
S2NB,2760.0,5911.0,7043.0,6733.0,7928.0,7153.0,7219.0,7295.0
S2SB,2817.0,9535.0,8678.0,8461.0,7027.0,6866.0,7013.0,5774.0
S3NB,2870.0,6148.0,7325.0,7003.0,8246.0,7439.0,7508.0,7587.0
S3SB,2929.0,9916.0,9025.0,8799.0,7308.0,7141.0,7293.0,6005.0
S4NB,2650.0,5675.0,6762.0,6463.0,7611.0,6867.0,6930.0,7003.0
S4SB,2704.0,9153.0,8331.0,8123.0,6746.0,6591.0,6732.0,5543.0
S5NB,1487.0,4065.0,4453.0,4266.0,4708.0,5570.0,5539.0,3830.0


In [100]:
# --- Lista de periodos según tus columnas ---
period_cols = ["Night","AM-Early","AM-Peak","AM-Shoulder","MD","PM-Shoulder","PM-Peak","PM-Late"]

# Diccionario de dataframes por clase
class_dfs = {
    "Lights": df_lights,
    "Medium A": df_mediumA,
    "Medium B": df_mediumB,
    "Heavy A": df_heavyA,
    "Heavy B": df_heavyB
}

projected_long_by_class = {}

for cls_name, df_class in class_dfs.items():
    df = df_class.copy()
    
    # Resetear índice Seg/Dir y extraer Segment y Direction
    df = df.reset_index()
    df["Segment"] = df["Seg/Dir"].str.extract(r"(\d+)")[0]    # solo los números
    df["Direction"] = df["Seg/Dir"].str.extract(r"([A-Z]+)")[0]  # solo las letras
    df["Class"] = cls_name
    
    # Melt usando las columnas de periodos
    df_long = df.melt(
        id_vars=["Seg/Dir","Segment","Direction","Class"],
        value_vars=period_cols,
        var_name="Period",
        value_name="AADT "+str(df["Class"][0])
    )
    
    # Normalizar SegDir (opcional)
    df_long["SegDir"] = df_long["Seg/Dir"].str.strip().str.upper().str.lstrip("S")
    
    projected_long_by_class[cls_name] = df_long

# Ejemplo: ver Lights
projected_long_lights_df = projected_long_by_class["Lights"]
projected_long_mediumA_df = projected_long_by_class["Medium A"]
projected_long_mediumB_df = projected_long_by_class["Medium B"]
projected_long_heaviesA_df = projected_long_by_class["Heavy A"]
projected_long_heaviesB_df = projected_long_by_class["Heavy B"]
projected_long_lights_df


,Seg/Dir,Segment,Direction,Class,Period,AADT Lights,SegDir
0,S1NB,1,S,Lights,Night,1830.0,1NB
1,S1SB,1,S,Lights,Night,1785.0,1SB
2,S2NB,2,S,Lights,Night,2760.0,2NB
3,S2SB,2,S,Lights,Night,2817.0,2SB
4,S3NB,3,S,Lights,Night,2870.0,3NB
...,...,...,...,...,...,...,...
107,S5SB,5,S,Lights,PM-Late,3555.0,5SB
108,S6NB,6,S,Lights,PM-Late,3740.0,6NB
109,S6SB,6,S,Lights,PM-Late,3472.0,6SB
110,S7NB,7,S,Lights,PM-Late,2375.0,7NB


In [101]:
rows = []

for year in years:
    for seg in seg_params.index:  # e.g., "1NB", "1SB", etc.
        seg_data = seg_params.loc[seg]
        # Extraer parte numérica y dirección
        seg_numeric = ''.join(filter(str.isdigit, seg))  # e.g., "10"
        direction = seg[len(seg_numeric):]       
        for p, hrs, peak, tag in period_template:
            rows.append({
                "Year": year,
                "SegDir": seg,
                "Segment": seg_numeric,        
                "Direction": direction,     
                "Period": p,
                "Hours/Day": hrs,
                "Peak": peak,
                "4Periods": tag,

                # Parámetros técnicos
                "Length": seg_data["Length"],
                "Speed GP": seg_data["Speed_GP"],
                "Capacity GP": seg_data["CapPerLane_GP"] * seg_data["Lanes_GP"],
                "Alpha GP": seg_data["Alpha_GP"],
                "Beta GP": seg_data["Beta_GP"],
                "Speed ML": seg_data["Speed_ML"],
                "Capacity ML": seg_data["CapPerLane_ML"] * seg_data["Lanes_ML"],
                "Alpha ML": seg_data["Alpha_ML"],
                "Beta ML": seg_data["Beta_ML"],
                "MinToll": 0.5,
                "MinCapture": 0
            })

# --- plantilla base ---
first_model_df = pd.DataFrame(rows)

# --- merge para todas las clases ---
for cls_name, df_proj in projected_long_by_class.items():
    proj_merge_df = df_proj[["SegDir", "Period", f"AADT {cls_name}"]].copy()
    proj_merge_df.rename(columns={"AADT": f"AADT {cls_name}"}, inplace=True)

    first_model_df = first_model_df.merge(
        proj_merge_df,
        on=["SegDir", "Period"],
        how="left"
    )

# --- 1. Reshape growths a formato largo ---
growths_long = base_growth_df.melt(
    id_vars="SegmentMapped",
    var_name="Year",
    value_name="AnnualGrowth"
).copy()
growths_long["Year"] = growths_long["Year"].astype(int)

# --- 2. Calcular crecimiento acumulado desde 2025 ---
# Ordenamos por año y aplicamos cumprod
growths_long = growths_long.sort_values(["SegmentMapped", "Year"])
growths_long["GrowthFactor"] = (growths_long["AnnualGrowth"]).groupby(growths_long["SegmentMapped"]).cumprod()

# Ahora GrowthFactor(y) = factor acumulado 2025→y

# --- 3. Preparar plantilla ---
fm = first_model_df.copy()
fm["Year"] = fm["Year"].astype(int)
fm["Segment"] = fm["Segment"].astype(str).str.replace(r"^S", "", regex=True)
fm["SegmentMapped"] = "S" + fm["Segment"].astype(str)

# --- 4. Merge GrowthFactor ---
fm = fm.merge(
    growths_long[["SegmentMapped", "Year", "GrowthFactor"]],
    on=["SegmentMapped", "Year"],
    how="left"
)

fm["GrowthFactor"] = fm["GrowthFactor"].fillna(1.0)

# --- 5. Aplicar GrowthFactor a todas las clases ---
for cls_name in projected_long_by_class.keys():
    col = f"AADT {cls_name}"
    if col in fm.columns:
        fm[col] = (fm[col].fillna(0) * fm["GrowthFactor"]).round(1)

# --- 6. Limpieza ---
fm = fm.drop(columns=["SegmentMapped"])   # opcional

first_model_df = fm

first_model_df


,Year,SegDir,Segment,Direction,Period,Hours/Day,Peak,4Periods,Length,Speed GP,...,Alpha ML,Beta ML,MinToll,MinCapture,AADT Lights,AADT Medium A,AADT Medium B,AADT Heavy A,AADT Heavy B,GrowthFactor
0,2025,1NB,1,NB,Night,8,OP,NT,3.0,65,...,1,6,0.5,0,1830.0,226.0,0.0,0.0,0.0,1.0
1,2025,1NB,1,NB,AM-Early,1,OP,AM,3.0,65,...,1,6,0.5,0,3467.0,429.0,0.0,0.0,0.0,1.0
2,2025,1NB,1,NB,AM-Peak,2,Peak,AM,3.0,65,...,1,6,0.5,0,4855.0,600.0,0.0,0.0,0.0,1.0
3,2025,1NB,1,NB,AM-Shoulder,1,OP,AM,3.0,65,...,1,6,0.5,0,4132.0,511.0,0.0,0.0,0.0,1.0
4,2025,1NB,1,NB,MD,5,OP,MD,3.0,65,...,1,6,0.5,0,5636.0,697.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,2025,7SB,7,SB,AM-Shoulder,1,OP,AM,5.6,70,...,1,6,0.5,0,2823.0,349.0,0.0,0.0,0.0,1.0
108,2025,7SB,7,SB,MD,5,OP,MD,5.6,70,...,1,6,0.5,0,2615.0,323.0,0.0,0.0,0.0,1.0
109,2025,7SB,7,SB,PM-Shoulder,1,OP,PM,5.6,70,...,1,6,0.5,0,2670.0,330.0,0.0,0.0,0.0,1.0
110,2025,7SB,7,SB,PM-Peak,3,Peak,PM,5.6,70,...,1,6,0.5,0,2493.0,308.0,0.0,0.0,0.0,1.0


In [102]:
first_model_df["Capacity GP"] = first_model_df.apply(
    lambda row: seg_params.loc[row["SegDir"], 'Cap_GP'],
    axis=1
)

first_model_df["B1"] = first_model_df.apply(
    lambda row: beta_vals[row["Year"]].loc[row["4Periods"], 1],
    axis=1
)

first_model_df["B2"] = first_model_df.apply(
    lambda row: beta_vals[row["Year"]].loc[row["4Periods"], 2],
    axis=1
)

# Here we load th value of the counts and we multiply the peak hour values by a constant
lights_w = 1

heavies_w = 3
heavies_w_toll = 5
heavies_w_vot = 5

medium_A_w = 3 #1.5 # TBD: Maybe try 2.5 or 2.75 for every pce value
medium_A_w_toll = 5
medium_A_w_vot = 5

medium_B_w = 3 #2.75
medium_B_w_toll = 5
medium_B_w_vot = 5

heavy_A_w = 3 # 2.75
heavy_A_w_toll = 5
heavy_A_w_vot = 5

heavy_B_w = 3
heavy_B_w_toll = 5
heavy_B_w_vot = 5

# 2. Compute TotalLights
first_model_df["TotalLights"] = first_model_df["AADT Lights"] 

first_model_df["TotalMediumA"] = first_model_df["AADT Medium A"]

first_model_df["TotalMediumB"] = first_model_df["AADT Medium B"]

first_model_df["TotalHeavyA"] = first_model_df["AADT Heavy A"]

first_model_df["TotalHeavyB"] = first_model_df["AADT Heavy B"]

first_model_df["TotalVeh"] = first_model_df.apply(
    lambda row: row["TotalLights"] + row["TotalMediumA"] + row["TotalMediumB"] + row["TotalHeavyA"] + row["TotalHeavyB"],
    axis=1
)

first_model_df["Corridor PCE pre-fix"] = first_model_df.apply(
    lambda row: row["TotalLights"] * lights_w + row["TotalMediumA"] * medium_A_w + row["TotalMediumB"] * medium_B_w + row["TotalHeavyA"] * heavy_A_w + row["TotalHeavyB"] * heavy_B_w,
    axis=1
)

first_model_df["Corridor PCE"] = first_model_df["Corridor PCE pre-fix"] # To anulate the suppression


first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalLights'] *= peak_factor

first_model_df["HOV3"] = first_model_df.apply(
    lambda row: row["TotalLights"] * hov_percentage.loc[row['Year']]['HOV percentage'],
    axis=1
)

first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalMediumA'] *= peak_factor
first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalMediumB'] *= peak_factor
first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalHeavyA'] *= peak_factor
first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalHeavyB'] *= peak_factor

first_model_df["TotalVeh"] = first_model_df.apply(
    lambda row: row["TotalLights"] + row["TotalMediumA"] + row["TotalMediumB"] + row["TotalHeavyA"] + row["TotalHeavyB"],
    axis=1
)

first_model_df["InScopeLights"] = first_model_df.apply(
    lambda row: (row["TotalLights"] - row["HOV3"]) * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeMediumA"] = first_model_df.apply(
    lambda row: row["TotalMediumA"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeMediumB"] = first_model_df.apply(
    lambda row: row["TotalMediumB"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeHeavyA"] = first_model_df.apply(
    lambda row: row["TotalHeavyA"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeHeavyB"] = first_model_df.apply(
    lambda row: row["TotalHeavyB"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeVeh"] = first_model_df.apply(
    lambda row: row["TotalVeh"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

# first_model_df.to_csv('model_test.csv')

In [103]:
def get_speed(row):

    gp_pce = (
        row["TotalLights"] * lights_w + 
        row["TotalMediumA"] * medium_A_w +
        row["TotalMediumB"] * medium_B_w +
        row["TotalHeavyA"] * heavy_A_w +
        row["TotalHeavyB"] * heavy_B_w
    )

    speedGP =  row["Speed GP"] / (1 + row["Alpha GP"] * ((gp_pce / row["Capacity GP"]) ** row["Beta GP"]))

    timeGP = 60 * row["Length"] / speedGP

    gp_vc = gp_pce / row["Capacity GP"]

    return pd.Series([speedGP, timeGP], index=["Speed GP","Time GP"])

In [104]:
first_model_df[["Speed GP Real", "Time GP"]] = first_model_df.apply(
    get_speed, axis=1, result_type='expand'
)

In [105]:
def get_capacity(row):

    measured_speed = measured_speeds.loc[row['Period']][row['SegDir']]

    gp_pce = (
        row["TotalLights"] * lights_w + 
        row["TotalMediumA"] * medium_A_w +
        row["TotalMediumB"] * medium_B_w +
        row["TotalHeavyA"] * heavy_A_w +
        row["TotalHeavyB"] * heavy_B_w
    )

    speed_rate = row['Speed GP'] / measured_speed

    if speed_rate < 1:

        measured_speed = row['Speed GP'] - 0.01

    capacity = gp_pce / ((row['Speed GP'] / (measured_speed * row['Alpha GP']) - 1 / row['Alpha GP']) ** (1 / row['Beta GP']))

    capacity_factor = capacity / row["Capacity GP"]

    return capacity_factor

In [106]:
first_model_df["Capacity Factor"] = first_model_df.apply(
    lambda row: get_capacity(row),
    axis=1
)

In [107]:
cap_factor = first_model_df.pivot(
    index=["Period"],
    columns="SegDir",
    values="Capacity Factor"
)

cap_factor.to_csv('inputs/capacity_factors.csv')

cap_factor

SegDir,1NB,1SB,2NB,2SB,3NB,3SB,4NB,4SB,5NB,5SB,6NB,6SB,7NB,7SB
Period,,,,,,,,,,,,,,
AM-Early,1.283552,2.721812,0.771502,1.356114,2.303795,1.352179,0.804419,1.259859,0.722439,2.406575,0.650695,0.910357,0.340169,2.042910
AM-Peak,1.886652,1.540546,0.856640,0.847064,0.920653,0.833056,0.859019,0.616555,0.764831,0.422079,0.636584,0.455582,0.390454,0.819305
AM-Shoulder,1.529517,0.889450,0.745617,0.703214,0.760322,0.722947,0.767227,0.559155,0.672673,0.434969,0.558601,0.432671,0.452515,0.571084
MD,2.086245,2.057896,0.819587,0.640740,0.837866,0.698279,0.898405,0.597058,0.778084,0.706934,0.609211,0.663692,0.614508,0.676845
Night,0.677145,0.660946,0.315023,0.383624,0.348511,0.413900,0.335710,0.351444,0.253391,0.297392,0.231935,0.233136,0.152387,0.206520
PM-Late,0.534228,1.407750,0.579360,0.701344,0.653449,0.663991,0.729669,0.614922,0.556310,0.645285,0.429372,0.492198,0.447067,0.434413
PM-Peak,0.502199,2.333155,0.545917,0.858318,0.626250,0.776245,0.680454,0.722441,0.680544,0.841186,0.503111,0.693898,0.677396,0.614735
PM-Shoulder,0.735031,2.379999,0.572848,0.666574,0.674318,0.769854,0.742972,0.737607,0.910988,0.885580,0.595724,0.688525,0.813888,0.651759


In [108]:
period_order = [
    "Night",
    "AM-Early",
    "AM-Peak",
    "AM-Shoulder",
    "MD",
    "PM-Shoulder",
    "PM-Peak",
    "PM-Late"
]

first_model_df["Total Corridor"] = first_model_df.apply(
    lambda row: row["Corridor PCE"] * row["Hours/Day"],
    axis=1
)


first_model_df["Period"] = pd.Categorical(
    first_model_df["Period"],
    categories=period_order,
    ordered=True
)

corridor_pce = first_model_df.pivot(
    index=["Period"],
    columns="SegDir",
    values="Total Corridor"
)

corridor_pce.to_csv('corridor_vals.csv')